# 20.12 排序学习 / Learning to Rank (RankNet & LambdaRank)

**中文**:搜索引擎、推荐、广告的核心任务不是"预测一个绝对分数",而是**把一堆候选按相关性排好序**——用户只看前几条,**排序的顺序就是一切**。**排序学习(Learning to Rank, LTR)** 专门解决这个问题:给定一个查询(query)和一批文档(documents),学习一个模型把最相关的排在最前面。它和普通回归/分类的关键区别:**我们不在乎预测分数的绝对值,只在乎相对顺序**,且评价指标(NDCG)**对排在前面的位置权重更高**。本节从零实现三大范式——**pointwise(逐点回归)**、**pairwise(RankNet,成对)**、**listwise 思想(LambdaRank,直接优化 NDCG)**,并诚实探讨:方法的精巧(如 LambdaRank 直接对准 NDCG)**在什么条件下才真正带来提升**。
**English**: The core task of search, recommendation, and ads is not "predict an absolute score" but **order a set of candidates by relevance** — users see only the top few, so **the ordering is everything**. **Learning to Rank (LTR)** solves exactly this: given a query and a set of documents, learn a model that puts the most relevant first. Its key difference from ordinary regression/classification: **we don't care about the absolute predicted values, only the relative order**, and the metric (NDCG) **weights top positions more heavily**. This section implements the three paradigms from scratch — **pointwise (regression)**, **pairwise (RankNet)**, and the **listwise idea (LambdaRank, directly optimizing NDCG)** — and honestly examines **under what conditions method sophistication (like LambdaRank aiming directly at NDCG) actually pays off**.

---

**中文**:**三大范式 / Three paradigms**:
**English**: **Three paradigms**:
- **中文**:**Pointwise(逐点)**:把每个文档的相关性当回归/分类目标独立预测(如预测 relevance∈{0..4}),再按预测分排序。简单,但**忽略了"排序是相对的"**——它努力拟合绝对分数,而非顺序。
  **Pointwise**: predict each document's relevance independently as regression/classification (e.g. relevance∈{0..4}), then sort by the prediction. Simple, but **ignores that ranking is relative** — it fits absolute scores rather than order.
- **中文**:**Pairwise(成对)**:看**文档对** $(i,j)$——如果 $i$ 比 $j$ 更相关,就学习让 $\text{score}(i)>\text{score}(j)$。**RankNet** 是代表:用 $\sigma(s_i-s_j)$ 建模"$i$ 排在 $j$ 前"的概率,交叉熵损失。直接学相对顺序。
  **Pairwise**: look at **document pairs** $(i,j)$ — if $i$ is more relevant than $j$, learn to make $\text{score}(i)>\text{score}(j)$. **RankNet** is the archetype: model the probability that "$i$ ranks above $j$" as $\sigma(s_i-s_j)$ with cross-entropy loss. Directly learns relative order.
- **中文**:**Listwise(列表)**:直接优化整个列表的排序指标(如 NDCG)。**LambdaRank/LambdaMART** 是里程碑:在 RankNet 的成对梯度上,**乘以"交换这对文档会导致 NDCG 变化多少"($|\Delta\text{NDCG}|$)** 作为权重——于是模型把力气花在**对指标影响最大的对**(通常是头部)上。
  **Listwise**: directly optimize a list-level ranking metric (e.g. NDCG). **LambdaRank/LambdaMART** are landmarks: multiply RankNet's pairwise gradient by **"how much NDCG changes if this pair is swapped" ($|\Delta\text{NDCG}|$)** as a weight — so the model spends effort on **the pairs that most affect the metric** (usually the top).

**中文**:**评价指标 NDCG(Normalized Discounted Cumulative Gain)**:
**English**: **The metric NDCG (Normalized Discounted Cumulative Gain)**:
$$\text{DCG@}k=\sum_{i=1}^{k}\frac{2^{rel_i}-1}{\log_2(i+1)},\qquad \text{NDCG@}k=\frac{\text{DCG@}k}{\text{IDCG@}k}$$
**中文**:分子:相关性高的文档贡献大($2^{rel}-1$),但**排得越靠后,除以的 $\log_2(i+1)$ 越大→贡献越小**(位置折扣,体现"用户只看前面")。分母 IDCG 是理想排序的 DCG,归一化到 [0,1]。**NDCG@1 只看第一名——最苛刻、最贴近"用户第一眼看到什么"**。
**English**: Numerator: highly relevant docs contribute more ($2^{rel}-1$), but **the lower the position, the larger the $\log_2(i+1)$ divisor → the smaller the contribution** (positional discount, reflecting "users only look at the top"). The denominator IDCG is the ideal ordering's DCG, normalizing to [0,1]. **NDCG@1 looks only at the first result — the strictest, closest to "what the user sees first."**

> 💡 **面试速查 / Interview cheat-sheet（★★★ 搜索/推荐/广告必考）**
> **中文**:**排序学习(LTR)**=学模型给"查询-文档"列表排序, 只在乎**相对顺序**不在乎绝对分。**三范式**:①**pointwise**(独立回归每个doc相关性→排序, 简单但忽略相对性)；②**pairwise**(学文档对顺序, **RankNet**用σ(sᵢ-sⱼ)+交叉熵)；③**listwise**(直接优化列表指标, **LambdaRank/LambdaMART**)。**LambdaRank 核心技巧**:成对梯度×**|ΔNDCG|**(交换这对导致的指标变化)→把学习力度集中在影响指标最大的头部对→**理论上在 NDCG@1/@3 等头部指标上更有优势**(但只在指标远未饱和的难问题上兑现;干净/简单数据上未必赢过 RankNet)。**LambdaMART**=LambdaRank+GBDT(梯度提升树), **工业界搜索排序长期SOTA/最常用**。**指标**:NDCG(位置折扣, 支持分级相关)、MAP(二元相关)、MRR(第一个相关的位置倒数)。**特征**:query-doc相关(BM25/TF-IDF/语义相似)、doc质量、点击/行为特征。**流程**:召回(recall, 海量→千)→粗排→精排(LTR)→重排(多样性/业务规则)。**注意**:位置偏差(用户偏爱靠前→用IPW/无偏LTR纠偏)、点击≠相关。
> **English**: **Learning to Rank (LTR)** = learn a model to order a "query-document" list, caring only about **relative order** not absolute scores. **Three paradigms**: ① **pointwise** (regress each doc's relevance independently → sort; simple but ignores relativity); ② **pairwise** (learn pair order; **RankNet** uses σ(sᵢ-sⱼ) + cross-entropy); ③ **listwise** (directly optimize a list metric; **LambdaRank/LambdaMART**). **LambdaRank's key trick**: pairwise gradient × **|ΔNDCG|** (the metric change from swapping that pair) → concentrate learning on the top pairs that most affect the metric → **theoretically advantageous on top metrics like NDCG@1/@3** (but only pays off on hard problems far from metric saturation; on clean/simple data it may not beat RankNet). **LambdaMART** = LambdaRank + GBDT (gradient-boosted trees), **long the industry SOTA/most-used for search ranking**. **Metrics**: NDCG (positional discount, supports graded relevance), MAP (binary relevance), MRR (reciprocal rank of first relevant). **Features**: query-doc relevance (BM25/TF-IDF/semantic), doc quality, click/behavior features. **Pipeline**: recall (millions→thousands) → coarse rank → fine rank (LTR) → re-rank (diversity/business rules). **Watch**: position bias (users favor top → correct with IPW/unbiased LTR), clicks ≠ relevance.


In [ ]:

# ============================================================
# ① NDCG 指标 + 合成 LTR 数据 / NDCG metric + synthetic LTR data
# 中文:造 Q 个查询, 每查询 30 个文档, 每文档有特征, 相关性分级 0..4(由一个全局非线性相关函数生成)。
# English: Q queries, each with 30 docs; each doc has features; graded relevance 0..4 (from a global nonlinear function).
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
ndoc, D = 30, 10
w_true=np.random.randn(D); V=np.random.randn(D,D)                 # 全局(共享)相关性函数 / global relevance fn
def relevance(X): return np.tanh(X@V)@w_true                       # 非线性隐相关分 / nonlinear latent relevance
def make(Q, seed):
    rng=np.random.default_rng(seed); Xs,rels=[],[]
    for q in range(Q):
        X=rng.standard_normal((ndoc,D)); s=relevance(X)+0.3*rng.standard_normal(ndoc)
        r=np.digitize(s, np.quantile(s,[0.5,0.75,0.9,0.97]))       # 分级相关性 0..4 / graded relevance
        Xs.append(X); rels.append(r)
    return Xs, rels
Xtr,Rtr=make(250,1); Xte,Rte=make(100,2)

def dcg(rel, k=10):
    rel=np.asarray(rel)[:k]
    return np.sum((2.0**rel - 1)/np.log2(np.arange(2, len(rel)+2)))  # Σ (2^rel-1)/log2(i+1)
def ndcg(rel_by_pred, rel_all, k=10):
    idcg=dcg(sorted(rel_all, reverse=True), k)                     # 理想排序的 DCG / ideal DCG
    return dcg(rel_by_pred, k)/idcg if idcg>0 else 0.0
def eval_ndcg(model, Xs, Rs, k=10):
    vals=[]
    for X,r in zip(Xs,Rs):
        s=model(torch.tensor(X,dtype=torch.float32)).detach().numpy().ravel()
        vals.append(ndcg(r[np.argsort(-s)], r, k))                 # 按预测分排序后的 NDCG / NDCG of predicted order
    return np.mean(vals)
print("示例查询的相关性分布 / example query relevance:", np.bincount(Rtr[0], minlength=5), "(0=无关 .. 4=最相关)")


In [ ]:

# ============================================================
# ② 三种排序模型 / three ranking models
# ============================================================
def mlp(): return nn.Sequential(nn.Linear(D,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))

# --- Pointwise:把相关性当回归目标 / regress the relevance grade ---
torch.manual_seed(1); pw=mlp(); opt=torch.optim.Adam(pw.parameters(),3e-3)
aX=torch.tensor(np.vstack(Xtr),dtype=torch.float32); aR=torch.tensor(np.concatenate(Rtr),dtype=torch.float32).view(-1,1)
for _ in range(400): opt.zero_grad(); F.mse_loss(pw(aX),aR).backward(); opt.step()

# --- Pairwise RankNet / listwise LambdaRank(共用一个训练函数, lambdarank=True 时加 |ΔNDCG| 权重) ---
def train_pairwise(lambdarank=False, epochs=40):
    torch.manual_seed(1); m=mlp(); opt=torch.optim.Adam(m.parameters(),3e-3)
    for ep in range(epochs):
        for X,r in zip(Xtr,Rtr):
            Xt=torch.tensor(X,dtype=torch.float32); s=m(Xt).ravel(); rt=torch.tensor(r,dtype=torch.float32)
            ii,jj=torch.where(rt.view(-1,1)>rt.view(1,-1))         # 所有 r_i>r_j 的文档对 / pairs with r_i>r_j
            if len(ii)==0: continue
            diff=s[ii]-s[jj]                                        # 分差 / score difference
            # RankNet 成对损失:P(i排j前)=σ(s_i-s_j), 目标=1 / pairwise BCE, target=1
            loss=F.binary_cross_entropy_with_logits(diff, torch.ones_like(diff), reduction='none')
            if lambdarank:
                # LambdaRank:乘 |ΔNDCG| —— 交换 i,j 会让 NDCG 变多少(聚焦头部)/ weight by NDCG change of swapping
                with torch.no_grad():
                    sn=s.detach().numpy(); order=np.argsort(-sn); rank=np.empty(len(order),int); rank[order]=np.arange(len(order))
                    idcg=dcg(sorted(r,reverse=True),len(r))+1e-9; gain=2.0**r-1; disc=1/np.log2(rank+2)
                    dndcg=np.abs((gain[ii.numpy()]-gain[jj.numpy()])*(disc[ii.numpy()]-disc[jj.numpy()]))/idcg
                loss=loss*torch.tensor(dndcg,dtype=torch.float32)
            opt.zero_grad(); loss.mean().backward(); opt.step()
    return m
rn=train_pairwise(lambdarank=False); lm=train_pairwise(lambdarank=True)

# --- 评估:多个 NDCG@k + 随机基线 / evaluate NDCG at several cutoffs + random baseline ---
rng=np.random.default_rng(0)
def rand_ndcg(k): return np.mean([ndcg(r[rng.permutation(len(r))], r, k) for r in Rte])
print(f"{'方法/method':<22}{'NDCG@1':>9}{'NDCG@3':>9}{'NDCG@5':>9}{'NDCG@10':>10}")
rows={"随机 random":lambda k: rand_ndcg(k),"Pointwise 回归":lambda k: eval_ndcg(pw,Xte,Rte,k),
      "RankNet 成对":lambda k: eval_ndcg(rn,Xte,Rte,k),"LambdaRank 列表":lambda k: eval_ndcg(lm,Xte,Rte,k)}
scores={}
for name,fn in rows.items():
    scores[name]=[fn(k) for k in (1,3,5,10)]
    print(f"{name:<22}{scores[name][0]:>9.3f}{scores[name][1]:>9.3f}{scores[name][2]:>9.3f}{scores[name][3]:>10.3f}")
print("\n关键:所有学习模型碾压随机(0.12→0.9+); 干净数据上三种学习方法接近, RankNet 略优, LambdaRank 优势未显现")


**中文**:结果讲了一个诚实的故事:**排序学习让 NDCG@1 从随机的 0.12 飙到 0.9+**——模型几乎总能把最相关的文档放到第一位,这才是排序真正的价值。但**三种学习方法彼此非常接近**(NDCG@10 都 0.95+),甚至**最朴素的 pointwise 回归都到了 0.91**,而理论上更精巧、直接对准 NDCG 的 LambdaRank **并没有赢过简单的 RankNet**。这个"反直觉"的结果本身就是重要一课,下面可视化后详细剖析原因。
**English**: The result tells an honest story: **learning to rank lifts NDCG@1 from random's 0.12 to 0.9+** — the model almost always places the most relevant doc first, which is ranking's real value. But **the three learned methods are very close** (NDCG@10 all 0.95+), and even **the plainest pointwise regression reaches 0.91**, while the theoretically more sophisticated LambdaRank (aimed directly at NDCG) **did not beat the simpler RankNet**. This "counterintuitive" result is itself an important lesson, dissected in detail after the visualization below.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,5))
ks=[1,3,5,10]; colors={"随机 random":"#BBBBBB","Pointwise 回归":"#DD8452","RankNet 成对":"#4C72B0","LambdaRank 列表":"#55A868"}
# ① NDCG@k 曲线 / NDCG@k curves
for name in scores:
    ax[0].plot(ks, scores[name], "o-", color=colors[name], lw=2, ms=7, label=name)
ax[0].set_xticks(ks); ax[0].set_xlabel("截断位置 k (NDCG@k)"); ax[0].set_ylabel("NDCG(越高越好)")
ax[0].set_title("排序学习 vs 随机:各方法 NDCG@k / LTR vs random"); ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)
# ② 头部放大:NDCG@1 各方法对比 / zoom on NDCG@1
names=list(scores.keys()); v1=[scores[n][0] for n in names]
bars=ax[1].bar(range(len(names)), v1, color=[colors[n] for n in names])
for b,v in zip(bars,v1): ax[1].text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=10, weight="bold")
ax[1].set_xticks(range(len(names))); ax[1].set_xticklabels([n.split()[0] for n in names], fontsize=9)
ax[1].set_ylabel("NDCG@1(最苛刻的头部指标)"); ax[1].set_ylim(0,1.05)
ax[1].set_title("头部 NDCG@1:三种学习方法接近, 均远超随机");
plt.tight_layout(); plt.savefig("/tmp/adv12_viz.png",dpi=80); plt.show()
print("左:所有学习方法(彩线)远高于随机(灰); 右:头部 NDCG@1 三种学习方法接近(干净数据上差距被抹平)")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **排序学习的第一课:顺序才是目标,不是绝对分**:随机排序 NDCG@1 只有 0.09,而学习后的模型达到 0.9+——同样的文档、同样的特征,**仅仅因为学会了"把最相关的放前面",体验天差地别**。这就是为什么搜索/推荐/广告的核心永远是排序:用户只看前几条,**第一名放对了,一切都对了**。指标 NDCG 用位置折扣($1/\log_2(i+1)$)精确编码了这一点——越靠前的位置越金贵。
2. **诚实的意外:在干净数据上,精巧的 LambdaRank 并没有赢**。LambdaRank 的巧妙在于:它在 RankNet 的成对梯度上乘 $|\Delta\text{NDCG}|$——**交换一对文档若严重伤害 NDCG(通常在头部),这对就获得更大的学习权重**,理论上应在头部指标(NDCG@1)上更强。但本例的实际结果是:**RankNet(0.928)≈ pointwise(0.909)> LambdaRank(0.904)**,LambdaRank 反而垫底!为什么?因为**本例数据太"干净"了**:特征直接、线性可分性强、相关性信号清晰,模型轻松就把 NDCG 顶到 0.95+ 的近饱和区。当**指标已经接近天花板,几乎没有"排错的头部对"可修**时,LambdaRank 的 $|\Delta\text{NDCG}|$ 加权就失去了用武之地,反而因为把大量梯度权重压到极少数对上,训练信号更嘈杂、略微掉点。**这是一个关键的工程直觉:方法的精巧只有在问题足够难、指标远未饱和时才兑现价值——在简单问题上,复杂损失常常打不过简单损失。**
3. **那 LambdaRank/LambdaMART 为何仍是工业霸主?因为真实排序远比这难**。真实场景里:①**相关性标注分级、主观、极度稀疏**(海量文档没标注),指标远达不到 0.95;②**位置偏差**——训练数据来自点击,用户天然偏爱靠前结果,**点击≠相关**,不纠偏模型会学到"位置"而非"相关性"(需 IPW/无偏 LTR);③**特征工程是胜负手**(BM25、语义相似、点击率、doc 质量……);④在线**多目标**(相关性 vs 多样性 vs 商业价值)。正是在这种"指标远未饱和、头部错误代价高昂"的真实环境里,LambdaRank 的头部聚焦才充分兑现——这也解释了为什么 **LambdaMART(LambdaRank+GBDT)长期是工业搜索排序的 SOTA**,而在我们这个玩具数据上它的优势被抹平了。**记住:选方法要匹配问题难度——别在简单问题上迷信复杂损失,也别在困难问题上低估直接优化指标的价值。**

**English**:
1. **LTR's first lesson: order is the goal, not absolute scores**: random ordering gives NDCG@1 of just 0.12, while the learned model reaches 0.9+ — same docs, same features, yet **merely learning to "put the most relevant first" transforms the experience**. This is why search/recommendation/ads are always about ranking: users see only the top few, so **get the first result right and everything is right**. NDCG's positional discount ($1/\log_2(i+1)$) encodes exactly this — earlier positions are more precious.
2. **An honest surprise: on clean data, the sophisticated LambdaRank did not win.** LambdaRank's cleverness is multiplying RankNet's pairwise gradient by $|\Delta\text{NDCG}|$ — **if swapping a pair badly hurts NDCG (usually at the top), that pair gets a larger learning weight**, so it should be stronger on top metrics (NDCG@1). But the actual result here is: **RankNet (0.928) ≈ pointwise (0.909) > LambdaRank (0.904)** — LambdaRank came last! Why? Because **this data is too "clean"**: features are direct, near-linearly separable, and the relevance signal is clear, so models easily push NDCG to the near-saturated 0.95+ region. When **the metric is already near its ceiling with almost no "mis-ordered top pairs" left to fix**, LambdaRank's $|\Delta\text{NDCG}|$ weighting has nothing to exploit and even slightly hurts by concentrating gradient weight on a few pairs, making the training signal noisier. **This is a key engineering intuition: method sophistication only pays off when the problem is hard enough and the metric is far from saturated — on easy problems, complex losses often lose to simple ones.**
3. **So why is LambdaRank/LambdaMART still the industry king? Because real ranking is far harder.** In reality: ① **relevance labels are graded, subjective, and extremely sparse** (most docs unlabeled), so the metric is nowhere near 0.95; ② **position bias** — training data comes from clicks, users inherently favor top results, so **clicks ≠ relevance**, and without correction the model learns "position" not "relevance" (needs IPW/unbiased LTR); ③ **feature engineering is decisive** (BM25, semantic similarity, click-through rate, doc quality…); ④ online **multi-objective** (relevance vs diversity vs business value). It is exactly in this "metric far from saturation, top errors costly" real environment that LambdaRank's top-focus fully pays off — which is why **LambdaMART (LambdaRank+GBDT) has long been the SOTA for industry search ranking**, even though its advantage was flattened on our toy data. **Remember: match the method to the problem's difficulty — don't fetishize complex losses on easy problems, and don't underestimate directly optimizing the metric on hard ones.**

> 💼 **实战视角 / Practical angle**
> **中文**:LTR 落地:①**搜索引擎/电商搜索**(精排用 LambdaMART 或深度 LTR)；②**推荐系统精排**(CTR/CVR 预估后按期望价值排序)；③**广告排序**(eCPM=出价×CTR)。工业标配:**LambdaMART**(LightGBM/XGBoost 都内置 `lambdarank`/`rank:ndcg` 目标, 强特征+树模型长期打得过深度模型)；深度 LTR(DLCM、SetRank、大厂精排网络)。落地要点:①**分级相关标注**(人工/点击日志), 注意**位置偏差**→无偏 LTR(IPW);②**指标对齐业务**——用 NDCG@头部 / 线上 A/B(CTR、转化、GMV);③**流水线思维**(召回→粗排→精排→重排), LTR 在精排;④**特征为王**(query-doc 匹配 + 行为 + 上下文);⑤评估用离线 NDCG/MAP + 线上 A/B 双保险。面试金句:*"排序学习只在乎相对顺序:pointwise 回归相关性、pairwise(RankNet)学文档对顺序、listwise(LambdaRank/LambdaMART)直接优化 NDCG——LambdaRank 用 |ΔNDCG| 加权成对梯度, 把力气花在影响头部指标最大的对上, 所以头部最强; 工业界搜索排序长期用 LambdaMART(GBDT), 还要处理位置偏差和点击≠相关。"*
> **English**: LTR in practice: ① **search engines / e-commerce search** (fine rank with LambdaMART or deep LTR); ② **recommender fine-ranking** (sort by expected value after CTR/CVR estimation); ③ **ad ranking** (eCPM = bid × CTR). Industry standard: **LambdaMART** (LightGBM/XGBoost both have built-in `lambdarank`/`rank:ndcg` objectives; strong features + tree models long outperformed deep models); deep LTR (DLCM, SetRank, large-scale ranking nets). Deployment keys: ① **graded relevance labels** (human/click logs), mind **position bias** → unbiased LTR (IPW); ② **align the metric with business** — NDCG at the top / online A/B (CTR, conversion, GMV); ③ **pipeline thinking** (recall → coarse → fine → re-rank), LTR is fine-ranking; ④ **features are king** (query-doc match + behavior + context); ⑤ evaluate with offline NDCG/MAP + online A/B as double insurance. Interview line: *"LTR cares only about relative order: pointwise regresses relevance, pairwise (RankNet) learns pair order, listwise (LambdaRank/LambdaMART) directly optimizes NDCG — LambdaRank weights the pairwise gradient by |ΔNDCG|, spending effort on the pairs that most affect the top metric, so it's strongest at the top; industry search ranking long used LambdaMART (GBDT), and must handle position bias and clicks ≠ relevance."*

---
### 小结 / Summary
- **中文**:排序学习只在乎相对顺序; 三范式:pointwise(回归)、pairwise(RankNet 成对)、listwise(LambdaRank 直接优化 NDCG)。
- **English**: LTR cares only about relative order; three paradigms: pointwise (regression), pairwise (RankNet), listwise (LambdaRank, directly optimizing NDCG).
- **中文**:NDCG 有位置折扣(头部更金贵); LambdaRank 用 |ΔNDCG| 加权成对梯度聚焦头部——但只在指标远未饱和的难问题上兑现价值(干净数据上打不过简单 RankNet)。
- **English**: NDCG has a positional discount (top positions more precious); LambdaRank weights pairs by |ΔNDCG| to focus on the top — but pays off only on hard problems far from metric saturation (on clean data it loses to simple RankNet).
- **中文**:工业界搜索排序长期用 LambdaMART(LambdaRank+GBDT); 真实难点是稀疏分级标注、位置偏差、点击≠相关、特征工程。
- **English**: Industry search ranking long used LambdaMART (LambdaRank+GBDT); real challenges are sparse graded labels, position bias, clicks ≠ relevance, and feature engineering.
